# Exercise 06 Solution — GPU Cross-Correlation

In [ ]:
!nvidia-smi

In [ ]:
%%writefile xcorr_solution.cu
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <cuda_runtime.h>
#include <cufft.h>

#define CUDA_CHECK(call) do { cudaError_t e=(call); \
    if(e!=cudaSuccess){fprintf(stderr,"CUDA %s:%d %s\n",__FILE__,__LINE__, \
    cudaGetErrorString(e));exit(1);}} while(0)
#define CUFFT_CHECK(call) do { cufftResult e=(call); \
    if(e!=CUFFT_SUCCESS){fprintf(stderr,"cuFFT %d at %s:%d\n",e,__FILE__,__LINE__);exit(1);}} while(0)

// PART 1 SOLUTION: element-wise cross-power spectrum for all pairs
__global__ void cross_power_all_pairs(
    const cufftComplex* X, cufftComplex* out, int N, int n_freq
) {
    int k    = blockIdx.x * blockDim.x + threadIdx.x;
    int pair = blockIdx.y;
    if (k >= n_freq || pair >= N*N) return;

    int i = pair / N;
    int j = pair % N;

    // Load FFT bins for neuron i and j
    cufftComplex xi = X[i * n_freq + k];
    cufftComplex xj = X[j * n_freq + k];

    // Cross-power: xi * conj(xj)
    cufftComplex result;
    result.x = xi.x * xj.x + xi.y * xj.y;  // Re(xi * conj(xj))
    result.y = xi.y * xj.x - xi.x * xj.y;  // Im(xi * conj(xj))

    out[pair * n_freq + k] = result;
}

// PART 2 SOLUTION: normalize
__global__ void normalize_xcorr(float* xcorr, int N, int T_bins, float scale) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx >= N * N * T_bins) return;
    xcorr[idx] *= scale;
}

int main(int argc, char** argv)
{
    int N_neurons = (argc>1) ? atoi(argv[1]) : 50;
    int T_bins    = (argc>2) ? atoi(argv[2]) : 2000;
    // Correlated group: neurons 0..N/2-1 share a 10 Hz rhythm
    int N_corr = N_neurons / 2;
    float rate = 0.04f;  // 4% background
    float rhythm_rate = 0.5f;  // 50% chance of firing during each 10 Hz cycle peak
    float rhythm_hz = 10.0f;   // Hz
    int n_freq  = T_bins / 2 + 1;
    int n_pairs = N_neurons * N_neurons;

    printf("Cross-correlation: %d neurons × %d bins\n", N_neurons, T_bins);
    printf("Correlated group: neurons 0..%d (share 10 Hz rhythm)\n", N_corr-1);

    // Memory check
    size_t mem_cross = (size_t)n_pairs * n_freq * sizeof(cufftComplex);
    printf("d_cross_power size: %.1f MB\n", mem_cross / 1e6f);

    // Generate synthetic spike trains with shared rhythm
    float* h_spikes = (float*)calloc((size_t)N_neurons * T_bins, sizeof(float));
    srand(42);
    // Global rhythm: fires at 10 Hz
    float dt_s = 0.001f;  // 1 ms bins
    for (int t = 0; t < T_bins; t++) {
        float t_s = t * dt_s;
        // Rhythm amplitude at this time
        float rhythm = 0.5f + 0.5f * sinf(2*M_PI * rhythm_hz * t_s);
        for (int i = 0; i < N_neurons; i++) {
            float p = rate;
            if (i < N_corr) p += rhythm_rate * rhythm * 0.02f;  // correlated addition
            if ((float)rand()/RAND_MAX < p) h_spikes[i * T_bins + t] = 1.0f;
        }
    }

    // Count spikes
    int total = 0;
    for (int i=0;i<N_neurons*T_bins;i++) total+=(int)h_spikes[i];
    printf("Total spikes: %d (%.1f%%)\n", total, 100.f*total/(N_neurons*T_bins));

    // GPU
    float *d_spikes;
    cufftComplex *d_fft, *d_cross, *d_xcorr_c;

    CUDA_CHECK(cudaMalloc(&d_spikes,(size_t)N_neurons*T_bins*sizeof(float)));
    CUDA_CHECK(cudaMalloc(&d_fft,  (size_t)N_neurons*n_freq*sizeof(cufftComplex)));
    CUDA_CHECK(cudaMalloc(&d_cross,(size_t)n_pairs*n_freq*sizeof(cufftComplex)));
    CUDA_CHECK(cudaMalloc(&d_xcorr_c,(size_t)n_pairs*T_bins*sizeof(cufftComplex)));
    CUDA_CHECK(cudaMemcpy(d_spikes,h_spikes,(size_t)N_neurons*T_bins*sizeof(float),
                          cudaMemcpyHostToDevice));

    // Step 1: forward FFT — N_neurons transforms of length T_bins
    cufftHandle plan_fwd;
    CUFFT_CHECK(cufftPlan1d(&plan_fwd, T_bins, CUFFT_R2C, N_neurons));
    CUFFT_CHECK(cufftExecR2C(plan_fwd, d_spikes, d_fft));

    // Step 2: cross-power for all pairs
    int thr = 256;
    int blk_freq = (n_freq + thr - 1) / thr;
    dim3 grid(blk_freq, n_pairs);
    cross_power_all_pairs<<<grid, thr>>>(d_fft, d_cross, N_neurons, n_freq);

    // Step 3: inverse FFT — n_pairs transforms of length T_bins
    cufftHandle plan_inv;
    CUFFT_CHECK(cufftPlan1d(&plan_inv, T_bins, CUFFT_C2C, n_pairs));
    CUFFT_CHECK(cufftExecC2C(plan_inv, d_cross, d_xcorr_c, CUFFT_INVERSE));

    // Step 4: copy back and extract real parts
    cufftComplex* h_xcorr_c = (cufftComplex*)malloc(
        (size_t)n_pairs * T_bins * sizeof(cufftComplex));
    CUDA_CHECK(cudaMemcpy(h_xcorr_c, d_xcorr_c,
                          (size_t)n_pairs*T_bins*sizeof(cufftComplex),
                          cudaMemcpyDeviceToHost));

    float norm = 1.0f / T_bins;

    // Save: auto-corr(0,0), within-group(0,1), cross-group(0,N_corr)
    FILE* f = fopen("xcorr_sol.txt", "w");
    fprintf(f, "# lag auto_00 within_01 cross_0_Ncorr\n");
    int p00 = 0*N_neurons+0, p01=0*N_neurons+1, p0N=0*N_neurons+N_corr;
    for (int t = 0; t < T_bins; t++) {
        int lag = (t <= T_bins/2) ? t : t - T_bins;  // circular shift
        fprintf(f, "%d %.6f %.6f %.6f\n",
                lag,
                h_xcorr_c[p00*T_bins+t].x * norm,
                h_xcorr_c[p01*T_bins+t].x * norm,
                h_xcorr_c[p0N*T_bins+t].x * norm);
    }
    fclose(f);

    CUFFT_CHECK(cufftDestroy(plan_fwd)); CUFFT_CHECK(cufftDestroy(plan_inv));
    cudaFree(d_spikes); cudaFree(d_fft); cudaFree(d_cross); cudaFree(d_xcorr_c);
    free(h_spikes); free(h_xcorr_c);
    return 0;
}

In [ ]:
!nvcc -O2 -o xcorr_solution xcorr_solution.cu -lcufft -lm && ./xcorr_solution 50 2000

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

data = np.loadtxt('xcorr_sol.txt')
lags, auto, within, cross_grp = data[:, 0], data[:, 1], data[:, 2], data[:, 3]

# Show only ±200 ms
mask = np.abs(lags) <= 200

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].bar(lags[mask], auto[mask], width=1, color='steelblue', alpha=0.7)
axes[0].set_title('Auto-Correlogram (neuron 0,0)', fontsize=11)
axes[0].set_xlabel('Lag (ms)'); axes[0].axvline(0,color='r',linestyle='--',lw=1)
axes[0].grid(True, alpha=0.3)

axes[1].bar(lags[mask], within[mask], width=1, color='orange', alpha=0.7)
axes[1].set_title('Within-Group Cross-Corr (0,1)\n10 Hz oscillation visible', fontsize=11)
axes[1].set_xlabel('Lag (ms)'); axes[1].axvline(0,color='r',linestyle='--',lw=1)
for lag_ms in [-100, 0, 100]:
    axes[1].axvline(lag_ms, color='gray', linestyle=':', alpha=0.5)
axes[1].grid(True, alpha=0.3)

axes[2].bar(lags[mask], cross_grp[mask], width=1, color='green', alpha=0.7)
axes[2].set_title('Cross-Group Cross-Corr (0, N/2)\n(should be flat)', fontsize=11)
axes[2].set_xlabel('Lag (ms)'); axes[2].axvline(0,color='r',linestyle='--',lw=1)
axes[2].grid(True, alpha=0.3)

plt.suptitle('GPU Cross-Correlation: Shared Rhythm Creates Within-Group Oscillations',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('xcorr_result.png', dpi=150, bbox_inches='tight')
plt.show()

## Answer Key

### Part 1 — Cross-power spectrum

```c
cufftComplex xi = X[i * n_freq + k];   // FFT of spike train i at bin k
cufftComplex xj = X[j * n_freq + k];   // FFT of spike train j at bin k
result.x = xi.x*xj.x + xi.y*xj.y;    // Re(xi × conj(xj))
result.y = xi.y*xj.x - xi.x*xj.y;    // Im(xi × conj(xj))
out[pair * n_freq + k] = result;
```

### Part 2 — Normalize

```c
xcorr[idx] *= scale;   // scale = 1.0f / T_bins
```

### Part 3 — Plan sizes

```c
cufftPlan1d(&plan_fwd, T_bins, CUFFT_R2C, N_neurons);  // forward: N_neurons transforms
cufftPlan1d(&plan_inv, T_bins, CUFFT_C2C, n_pairs);    // inverse: n_pairs transforms
dim3 grid((n_freq + thr - 1) / thr, n_pairs);          // 2D grid for cross-power
```

### Reflection Answers

1. **Memory:** `n_pairs × n_freq × 8 bytes = 50² × 1001 × 8 ≈ 20 MB`. For 16 GB GPU, max N where cross-power fits: `16e9 / (8 × (T/2+1)) → N ≈ sqrt(16e9 / 8008) ≈ 1413`. So ~1400 neurons max for T=2000.

2. **Complexity breakeven:** FFT: O(N² × T log T). Direct: O(N² × r×T × τ_max/dt) where r is firing rate. FFT wins when `T log T < r × T × τ_max`, i.e., `log T < r × τ_max`. For r=0.05, τ_max=200 ms, dt=1 ms: `log T < 0.05×200 = 10 → T < e^10 ≈ 22,000`. FFT wins for T ≥ 22,000 bins.

3. **Normalization to [-1,1]:** Divide C_ij(τ) by √(C_ii(0) × C_jj(0)). This requires the auto-correlation at lag 0, which is already in the output for pairs (i,i).

4. **Lower triangle only:** Replace `n_pairs = N*N` with `n_pairs = N*(N+1)/2` and use an index mapping `pair → (i,j) with i≤j`. This halves memory and computation.